# Original Smart Bin / EcoRecycle 2025 notebook — outputs stripped

The code cells below transcribe the supplied 2025 `file.ipynb` source, with output cells omitted to avoid large unrelated image output. Its source configuration used YOLOv8m. The later inference/validation cells **reload generic pretrained weights**, so these results cannot be presented as custom-detector performance.


In [ ]:
import random
from PIL import Image, ImageDraw, ImageFont
import os

# Verify labels 

In [ ]:
def visualize_sample(image_path, label_path):
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.load_default()
    except:
        font = None

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, x_center, y_center, bw, bh = parts
            cls = int(cls)
            x_center, y_center, bw, bh = map(float, (x_center, y_center, bw, bh))
            x0 = (x_center - bw/2) * w
            y0 = (y_center - bh/2) * h
            x1 = (x_center + bw/2) * w
            y1 = (y_center + bh/2) * h
            draw.rectangle([x0, y0, x1, y1], outline="red", width=2)
            label_text = "plastic"
            if font:
                draw.text((x0, y0), label_text, fill="red", font=font)
            else:
                draw.text((x0, y0), label_text, fill="red")
    img.show()

In [ ]:
# Pick a few random samples
train_images = os.listdir('train/images')
for _ in range(3):
    img_name = random.choice(train_images)
    img_path = os.path.join('train/images', img_name)
    base = os.path.splitext(img_name)[0]
    label_path = os.path.join('train/labels', base + '.txt')
    visualize_sample(img_path, label_path)

# Model training (change to yolov8s and tweak batch size if it takes too long or memeory issues ) 

In [ ]:
from ultralytics import YOLO

# Create a YOLO model from pretrained medium weights
model = YOLO('yolov8m.pt')

# Train
results = model.train(data='data.yaml', epochs=50, imgsz=640, batch=16)

In [ ]:
#yolo detect val data=data.yaml model=runs/detect/train/weights/best.pt (to run validation )
#yolo detect predict model=runs/detect/train/weights/best.pt source=test/images save=True (test on samples)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')
# Inference on a single image
results = model.predict(source='test/images/WIN_20250611_16_05_09_Pro_jpg.rf.7d2c94f47bc45cd52c0ffe624e0552f3.jpg', conf=0.25, iou=0.45)
# results[0].boxes gives bounding boxes, confidences, class ids (should all be class 0)
# You can parse results to get box coordinates and confidence scores:
for r in results:
    for box in r.boxes:
        # box.xyxy, box.conf, box.cls
        coords = box.xyxy.cpu().numpy()  # [x1,y1,x2,y2]
        conf = box.conf.cpu().item()
        cls_id = int(box.cls.cpu().item())  # should be 0
        print(f"Detected plastic bottle at {coords} with confidence {conf:.2f}")

In [ ]:
#yolo detect tune data=data.yaml model=yolov8m.pt epochs=50 (params tuning better results)
#yolo export model=runs/detect/train/weights/best.pt format=onnx (export model)

In [ ]:
model = YOLO("yolov8m.pt")

results = model.val(data="data.yaml")
print(f"mAP@0.5: {results.box.map:.3f}")
print(f"mAP@0.5:0.95: {results.box.map50to95:.3f}")
print(f"Precision: {results.box.precision:.3f}")
print(f"Recall: {results.box.recall:.3f}")

In [ ]:
results = model.predict(source="test/images", save=True, conf=0.25)

In [ ]:
import os
from PIL import Image
output_folder = "runs/detect/predict"
for fname in os.listdir(output_folder)[:15]:
    display(Image.open(os.path.join(output_folder, fname))) 